Model selection

In [2]:
# load libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as sm

netMHC1 default settings (peptide length 9, window 5, human 27 allele panel). Yielded 1 warning:\
2 duplicates found of input sequence: "QVQLVESGGGVVQPGRSLRLSCAASGFTFSSYGMHWVRQAPGKGLEWVAVIWYDGSNKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDPRGATLYYYYYGMDVWGQGTTVTVSSDIQMTQSPSSLSASVGDRVTITCRASQSINSYLDWYQQKPGKAPKLLIYAASSLQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQYYSTPFTFGPGTKVEIK"

In [3]:
# load all data sets

# netMHC1 predictions
    # deafault settings (peptide length 9, window 5, human 27 allele panel)
    # separated into 6 predictions bc of issues with running full fasta file
netMHC1_EL_pep9_subset1 = pd.read_csv("tool_outputs/217AB_netMHC_peplen9_subset1.csv")

    # peptide length 14, window 5, human 27 allele panel
    # separated into 6 predictions bc of issues with running full fasta file
#netMHC1_EL_pep14_subset1 # not yet predicted

# netMHC II predicitons
    # default settings (peptide length 15, window 5, human 27 allele panel)
    # separated into 6 predictions bc of issues with running full fasta file
#netMHC_II_EL_pep15_subset1 # not yet predicted

    # peptide length 12, window 5, human 27 allele panel
    # separated into 6 predictions bc of issues with running full fasta file
#netMHC_II_EL_pep12_subset1 # not yet predicted

# sequence table from one of the netMHC predicitons. 
# The tools remove the antibody name and just put a number instead. This is so that the name can be mapped back later
# seperated into 6 predictions (with different antibodies in each) 
seqTable_subset1 = pd.read_csv("tool_outputs/seqTable217AB_netMHC_peplen9_subset1.csv")


# waltz predictions
# separated into 6 predicitons due to max capacity of waltz tool
waltz1 = pd.read_csv("tool_outputs/waltz1_217AB.txt", sep="\t", header=None, names=["antibody", "waltz_score", "..."])
waltz2 = pd.read_csv("tool_outputs/waltz2_217AB.txt", sep="\t", header=None, names=["antibody", "waltz_score", "..."])
waltz3 = pd.read_csv("tool_outputs/waltz3_217AB.txt", sep="\t", header=None, names=["antibody", "waltz_score", "..."])
waltz4 = pd.read_csv("tool_outputs/waltz4_217AB.txt", sep="\t", header=None, names=["antibody", "waltz_score", "..."])
waltz5 = pd.read_csv("tool_outputs/waltz5_217AB.txt", sep="\t", header=None, names=["antibody", "waltz_score", "..."])
waltz6 = pd.read_csv("tool_outputs/waltz6_217AB.txt", sep="\t", header=None, names=["antibody", "waltz_score", "..."])

# biophi predictions
    # default settings (numbering scheme Kabat, CDR definition Kabat, OASis prevalence threshold Relaxed)
    # seperated into 3 predicitons due to max capacity of biobhi humanness tool
biophi_Relaxed1 = pd.read_excel("tool_outputs/NRKab_CDRKab_Relaxed_pred1_217AB.xlsx")
biophi_Relaxed2 = pd.read_excel("tool_outputs/NRKab_CDRKab_Relaxed_pred2_217AB.xlsx")
biophi_Relaxed3 = pd.read_excel("tool_outputs/NRKab_CDRKab_Relaxed_pred3_217AB.xlsx")

    # numbering scheme Kabat, CDR definition Kabat, OASis prevalence threshold Strict
    # seperated into 3 predicitons due to max capacity of biobhi humanness tool
biophi_Strict1 = pd.read_excel("tool_outputs/NRKab_CDRKab_Strict_pred1_217AB.xlsx")
biophi_Strict2 = pd.read_excel("tool_outputs/NRKab_CDRKab_Strict_pred2_217AB.xlsx")
biophi_Strict3 = pd.read_excel("tool_outputs/NRKab_CDRKab_Strict_pred3_217AB.xlsx")


In [4]:
# load ADA scores
ADA = pd.read_csv("ADA_Clinical_Ab_2021_biophiDataset.csv")
# extract only the ADA (here called Immunogenicity) and Antibody (name) columns
ADA = ADA[["Antibody", "Immunogenicity"]].copy()
# make all antibody names lowercase
ADA["Antibody"] = ADA["Antibody"].str.lower()


# load another dataset with antibodies, that have been predicited previously
previous_37AB = pd.read_csv("../Antibodies/ADA_combined_features.csv")
# rename antibody column so it maches the one in the ADA df
previous_37AB = previous_37AB.rename(columns={'antibody': 'Antibody'})
# make all antibody names lowercase
previous_37AB["Antibody"] = previous_37AB["Antibody"].str.lower()


In [5]:
# Check how many/ which antibody names match in the ADA score table and in the previously predicted antibodies
# select antibody and ADA precentage column, rename so it matches ADA df
subset = previous_37AB[['Antibody', 'ADA_percentage']].rename(
    columns={'ADA_percentage': 'Immunogenicity'}
)

# collect the antibodies that do not exist in the ADA df
missing = subset[~subset['Antibody'].isin(ADA['Antibody'])]

# append to ADA df
ADA = pd.concat([ADA, missing], ignore_index=True)



In [ ]:
# Merge netMHC1 pep 9 dfs together

# Merge netMHC1 pep 14 dfs together

# Merge netMHC II pep 15 dfs together

# Merge netMHC II pep 12 dfs together


# Merge waltz dfs together
waltz = pd.concat([waltz1, waltz2, waltz3, waltz4, waltz5, waltz6], ignore_index=True)

# Merge biophi Relaxed dfs together
biophi_relaxed = pd.concat([biophi_Relaxed1, biophi_Relaxed2, biophi_Relaxed3], ignore_index=True)

# Merge biophi Strict dfs together
biophi_strict = pd.concat([biophi_Strict1, biophi_Strict2, biophi_Strict3], ignore_index=True)

In [7]:
# the netMHC tools do not have the antibody name in the output, only a sequence number
# Here the seq # will be mapped back to the original antibody name using the seqTable
netMHC1_EL_pep9 = netMHC1_EL_pep9.merge(seqTable[['seq #', 'sequence name']], how='left')

NameError: name 'netMHC1_EL_pep9' is not defined

In [ ]:
# Sanity checks

# Check nr of antibodies
if netMHC1_EL_pep14['seq #'].nunique()!=217:
        print( "netMHC1_EL_pep14 does not have 217 antibodies")
if netMHC1_EL_pep9['seq #'].nunique()!=217:
        print( "netMHC1_EL_pep9 does not have 217 antibodies")
if netMHC_II_EL_pep12['seq #'].nunique()!=217:
        print( "netMHC_II_EL_pep12 does not have 217 antibodies")   
if netMHC_II_EL_pep15['seq #'].nunique()!=217:
        print( "netMHC_II_EL_pep15 does not have 217 antibodies")
if biophi_KabKabRelaxed['Antibody'].nunique()!=217:
        print( "biophi_KabKabRelaxed does not have 217 antibodies")
if biophi_KabKabStrict['Antibody'].nunique()!=217:
        print( "biophi_KabKabStrict does not have 217 antibodies")
if waltz['antibody'].nunique()!=217:
        print( "waltz does not have 217 antibodies")

In [11]:
# Export merged dfs and ADA df, to use for model selection



waltz.to_csv("merged_tool_outputs/waltz_merged_tool_outputs.csv")
biophi_relaxed.to_csv("merged_tool_outputs/biophi_relaxed_merged_tool_outputs.csv")
biophi_strict.to_csv("merged_tool_outputs/biophi_strict_merged_tool_outputs.csv")

ADA.to_csv("ADA_218AB.csv", index=False)
